# PSELDNets — Colab セットアップ

## ⚠️ 使用前に必ず確認

1. **ランタイム → ランタイムを切断して削除** で完全にリセット
2. **ランタイム → ランタイムのタイプを変更 → T4 GPU** を選択
3. セルを **上から順に** 実行（途中でスキップしない）
4. **再起動は不要です**

---
## 1. GPU 確認

In [ ]:
import torch
assert torch.cuda.is_available(), '⚠️ GPU未接続。ランタイムのタイプを T4 GPU に変更してください。'
print(f'PyTorch : {torch.__version__}')
print(f'GPU     : {torch.cuda.get_device_name(0)}')
print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch : 2.11.0+cu128
GPU     : Tesla T4
VRAM    : 15.6 GB


## 2. リポジトリ clone

In [ ]:
import os

REPO = '/content/PSELDNets'

if not os.path.exists(f'{REPO}/src'):
    !git clone https://github.com/Jinbo-Hu/PSELDNets {REPO}
else:
    print(f'既にあります: {REPO}')

os.chdir(REPO)
print(f'CWD: {os.getcwd()}')
!ls

既にあります: /content/PSELDNets
CWD: /content/PSELDNets
AGG_LOSS.md  configs   _hdf5  LICENSE  README.md	 scripts
ckpts	     datasets  image  logs     requirements.txt  src


## 3. パッケージインストール

`numpy` / `h5py` / `scipy` / `torch` は Colab に最初から入っています。  
ここでは**触らず**、不足しているものだけ追加します（再起動不要）。

In [ ]:
!pip install -q \
    librosa \
    soundfile \
    lightning==2.2.1 \
    hydra-core==1.3.2 \
    hydra-colorlog==1.2.0 \
    hydra-joblib-launcher==1.2.0 \
    torchmetrics==1.3.1

print('完了')

完了


## 4. 動作確認

In [ ]:
import numpy as np, h5py, lightning, torchmetrics, librosa
print(f'numpy       : {np.__version__}')
print(f'h5py        : {h5py.__version__}')
print(f'lightning   : {lightning.__version__}')
print(f'torchmetrics: {torchmetrics.__version__}')
print(f'librosa     : {librosa.__version__}')
print('すべて OK')

numpy       : 2.0.2
h5py        : 3.16.0
lightning   : 2.2.1
torchmetrics: 1.3.1
librosa     : 0.11.0
すべて OK


---
## 5. チェックポイントのダウンロード

In [ ]:
import shutil
from huggingface_hub import hf_hub_download

os.makedirs('ckpts', exist_ok=True)
CKPT = 'ckpts/mACCDOA-HTSAT-0.567.ckpt'

if not os.path.exists(CKPT):
    print('ダウンロード中...')
    src = hf_hub_download(
        repo_id='Jinbo-HU/PSELDNets',
        filename='model/mACCDOA-HTSAT-0.567.ckpt',
        repo_type='dataset',
    )
    shutil.copy(src, CKPT)

print(f'OK: {CKPT}  ({os.path.getsize(CKPT)/1e6:.0f} MB)')

OK: ckpts/mACCDOA-HTSAT-0.567.ckpt  (141 MB)


## 6. クラス定義ファイル (TSV)

In [ ]:
os.makedirs('datasets', exist_ok=True)

for tsv in ['cls_indices_train.tsv', 'cls_indices_test.tsv']:
    dest = f'datasets/{tsv}'
    if not os.path.exists(dest):
        src = hf_hub_download(
            repo_id='Jinbo-HU/PSELDNets',
            filename=f'dataset/{tsv}',
            repo_type='dataset',
        )
        shutil.copy(src, dest)
    print(f'OK: {dest}')

OK: datasets/cls_indices_train.tsv
OK: datasets/cls_indices_test.tsv


## 7. テストデータのダウンロード（約4GB）

In [ ]:
import zipfile

ZIP = 'datasets/test360_ov3.zip'
DATA_DIR = 'datasets/test360_ov3'

if not os.path.exists(f'{DATA_DIR}/foa'):
    if not os.path.exists(ZIP):
        print('ダウンロード中（約4GB）...')
        src = hf_hub_download(
            repo_id='Jinbo-HU/PSELDNets',
            filename='dataset/test360_ov3.zip',
            repo_type='dataset',
        )
        shutil.copy(src, ZIP)
        print(f'ダウンロード完了: {os.path.getsize(ZIP)/1e9:.2f} GB')

    print('解凍中...')
    with zipfile.ZipFile(ZIP) as z:
        z.extractall('datasets/')
    print('解凍完了')
else:
    print(f'既にあります: {DATA_DIR}')

!ls datasets/test360_ov3/

既にあります: datasets/test360_ov3
foa  metadata  mic  sum


---
## 8. 前処理（FLAC → HDF5 特徴量）

数分かかります。

In [ ]:
if not os.path.exists('_hdf5') or len(os.listdir('_hdf5')) == 0:
    !python src/preproc.py dataset=test360_ov3
else:
    print('既に前処理済み')
    !ls _hdf5/

既に前処理済み
data  label


## 9. 推論実行

In [ ]:
# 事前確認
checks = [
    ('ckpts/mACCDOA-HTSAT-0.567.ckpt', 'チェックポイント'),
    ('datasets/cls_indices_train.tsv',  'TSV train'),
    ('datasets/cls_indices_test.tsv',   'TSV test'),
    ('datasets/test360_ov3/foa',        'FOA データ'),
    ('_hdf5',                           'HDF5 特徴量'),
]
for path, name in checks:
    ok = os.path.exists(path) and (not os.path.isdir(path) or len(os.listdir(path)) > 0)
    print(f'  [{"OK" if ok else "NG"}] {name}')

  [OK] チェックポイント
  [OK] TSV train
  [OK] TSV test
  [OK] FOA データ
  [OK] HDF5 特徴量


In [ ]:
!python src/infer.py \
    experiment=synth_maccdoa \
    ckpt_path=ckpts/mACCDOA-HTSAT-0.567.ckpt \
    model.kwargs.pretrained_path=null

[2026-06-02 14:33:22,876][utils.utilities][INFO] - Printing config tree with Rich! <cfg.extras.print_config=True>
CONFIG
├── data
│   └── audio_type: foa                                                         
│       audio_feature: logmelIV                                                 
│       sample_rate: 24000                                                      
│       nfft: 1024                                                              
│       n_mels: 64                                                              
│       hoplen: 240                                                             
│       window: hann                                                            
│       train_chunklen_sec: 10                                                  
│       train_hoplen_sec: 10                                                    
│       test_chunklen_sec: 10                                                   
│       test_hoplen_sec: 10                                 

---
## 10. DCASE2021 ファインチューニング（テンプレート）

データは [Zenodo](https://zenodo.org/records/5476980) から取得し、`datasets/DCASE2021/` に配置してください。

```
datasets/DCASE2021/
├── foa_dev/
├── foa_eval/
├── metadata_dev/
└── metadata_eval/
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# データのダウンロード
import os

DCASE = 'datasets/DCASE2021'
os.makedirs(DCASE, exist_ok=True)

BASE = 'https://zenodo.org/records/5476980/files'

files = [
    'foa_dev.zip',       # 1.4 GB（分割 zip のメイン部分）
    'foa_dev.z01',       # 4.3 GB（分割 zip のパート 1）
    'foa_eval.zip',      # 1.9 GB
    'metadata_dev.zip',  # 2 MB
    'metadata_eval.zip', # 660 kB
]

for fname in files:
    dest = f'{DCASE}/{fname}'
    if not os.path.exists(dest):
        print(f'ダウンロード中: {fname}')
        !wget -q --show-progress -O {dest} {BASE}/{fname}?download=1
    else:
        print(f'スキップ（既にあります）: {fname}')

print('\nダウンロード完了')
!ls -lh {DCASE}/

ダウンロード中: foa_dev.zip
datasets/DCASE2021/ 100%[===================>]   1.33G  2.78MB/s    in 8m 22s  
ダウンロード中: foa_dev.z01
datasets/DCASE2021/ 100%[===================>]   4.00G  17.6MB/s    in 13m 25s 
ダウンロード中: foa_eval.zip
datasets/DCASE2021/ 100%[===================>]   1.76G  21.2MB/s    in 5m 7s   
ダウンロード中: metadata_dev.zip
datasets/DCASE2021/ 100%[===================>]   1.84M   614KB/s    in 3.2s    
ダウンロード中: metadata_eval.zip
datasets/DCASE2021/ 100%[===================>] 644.92K   594KB/s    in 1.1s    

ダウンロード完了
total 7.1G
-rw-r--r-- 1 root root 4.0G Jun  2 14:55 foa_dev.z01
-rw-r--r-- 1 root root 1.4G Jun  2 14:41 foa_dev.zip
-rw-r--r-- 1 root root 1.8G Jun  2 15:00 foa_eval.zip
-rw-r--r-- 1 root root 1.9M Jun  2 15:00 metadata_dev.zip
-rw-r--r-- 1 root root 645K Jun  2 15:00 metadata_eval.zip


In [ ]:
# 解凍&ディレクトリ整理
DCASE = 'datasets/DCASE2021'

# foa_dev: 分割 zip を結合してから解凍
if not os.path.exists(f'{DCASE}/foa_dev'):
    print('foa_dev を結合・解凍中（数分かかります）...')
    !zip -s 0 {DCASE}/foa_dev.zip --out {DCASE}/foa_dev_agg.zip
    !unzip -q {DCASE}/foa_dev_agg.zip -d {DCASE}/
    !rm {DCASE}/foa_dev_agg.zip
    print('foa_dev 完了')

if not os.path.exists(f'{DCASE}/foa_eval'):
    print('foa_eval を解凍中...')
    !unzip -q {DCASE}/foa_eval.zip -d {DCASE}/

if not os.path.exists(f'{DCASE}/metadata_dev'):
    !unzip -q {DCASE}/metadata_dev.zip -d {DCASE}/

if not os.path.exists(f'{DCASE}/metadata_eval'):
    !unzip -q {DCASE}/metadata_eval.zip -d {DCASE}/

# 解凍後はサブフォルダに wav/csv が入っているので平坦化する
print('ディレクトリ整理中...')
for folder, ext in [('foa_dev', 'wav'), ('foa_eval', 'wav'),
                    ('metadata_dev', 'csv'), ('metadata_eval', 'csv')]:
    path = f'{DCASE}/{folder}'
    !find {path} -mindepth 2 -name "*.{ext}" -exec mv -t {path}/ {{}} + 2>/dev/null || true
    !find {path} -mindepth 1 -maxdepth 1 -type d -exec rm -rf {{}} + 2>/dev/null || true

print('\n整理後のファイル数:')
for d in ['foa_dev', 'foa_eval', 'metadata_dev', 'metadata_eval']:
    n = len(os.listdir(f'{DCASE}/{d}')) if os.path.exists(f'{DCASE}/{d}') else 0
    print(f'  {d}/: {n} ファイル')

foa_dev を結合・解凍中（数分かかります）...
 copying: foa_dev/
 copying: foa_dev/dev-test/
 copying: foa_dev/dev-test/fold6_room1_mix001.wav
 copying: foa_dev/dev-test/fold6_room1_mix002.wav
 copying: foa_dev/dev-test/fold6_room1_mix003.wav
 copying: foa_dev/dev-test/fold6_room1_mix004.wav
 copying: foa_dev/dev-test/fold6_room1_mix005.wav
 copying: foa_dev/dev-test/fold6_room1_mix006.wav
 copying: foa_dev/dev-test/fold6_room1_mix007.wav
 copying: foa_dev/dev-test/fold6_room1_mix008.wav
 copying: foa_dev/dev-test/fold6_room1_mix009.wav
 copying: foa_dev/dev-test/fold6_room1_mix010.wav
 copying: foa_dev/dev-test/fold6_room1_mix011.wav
 copying: foa_dev/dev-test/fold6_room1_mix012.wav
 copying: foa_dev/dev-test/fold6_room1_mix013.wav
 copying: foa_dev/dev-test/fold6_room1_mix014.wav
 copying: foa_dev/dev-test/fold6_room1_mix015.wav
 copying: foa_dev/dev-test/fold6_room1_mix016.wav
 copying: foa_dev/dev-test/fold6_room1_mix017.wav
 copying: foa_dev/dev-test/fold6_room1_mix018.wav
 copying: foa_dev/dev-tes

In [ ]:
# 前処理
# 訓練データの前処理
!python src/preproc.py dataset=DCASE2021 wav_format=.wav

# 評価データの前処理
!python src/preproc.py dataset=DCASE2021 dataset_type=eval wav_format=.wav


Dataset DCASE2021 is being developed......

Extract indices for wav file: 1, fold1_room1_mix001.wav
Extract indices for wav file: 2, fold1_room1_mix002.wav
Extract indices for wav file: 3, fold1_room1_mix003.wav
Extract indices for wav file: 4, fold1_room1_mix004.wav
Extract indices for wav file: 5, fold1_room1_mix005.wav
Extract indices for wav file: 6, fold1_room1_mix006.wav
Extract indices for wav file: 7, fold1_room1_mix007.wav
Extract indices for wav file: 8, fold1_room1_mix008.wav
Extract indices for wav file: 9, fold1_room1_mix009.wav
Extract indices for wav file: 10, fold1_room1_mix010.wav
Extract indices for wav file: 11, fold1_room1_mix011.wav
Extract indices for wav file: 12, fold1_room1_mix012.wav
Extract indices for wav file: 13, fold1_room1_mix013.wav
Extract indices for wav file: 14, fold1_room1_mix014.wav
Extract indices for wav file: 15, fold1_room1_mix015.wav
Extract indices for wav file: 16, fold1_room1_mix016.wav
Extract indices for wav file: 17, fold1_room1_mix017

In [ ]:
# ファインチューニング
# !python src/train.py experiment=dcase2021/finetune_maccdoa_augmix1 model.batch_size=8

# メモリ不足時はバッチサイズを下げる
# !python src/train.py experiment=dcase2021/finetune_maccdoa_augmix1 model.batch_size=8 ckpt_path=ckpts/resume.ckpt

In [ ]:
## 14. 再開用（続きから学習する場合）
import shutil
shutil.copy('/content/drive/MyDrive/PSELDNets_ckpts/last.ckpt', 'ckpts/resume.ckpt')
print('OK')

!python src/train.py experiment=dcase2021/finetune_maccdoa_augmix1 \
    model.batch_size=8 \
    ckpt_path=ckpts/resume.ckpt

OK
[2026-06-02 15:04:21,930][utils.utilities][INFO] - Printing config tree with Rich! <cfg.extras.print_config=True>
CONFIG
├── data
│   └── audio_type: foa                                                         
│       audio_feature: logmelIV                                                 
│       sample_rate: 24000                                                      
│       nfft: 1024                                                              
│       n_mels: 64                                                              
│       hoplen: 240                                                             
│       window: hann                                                            
│       train_chunklen_sec: 10                                                  
│       train_hoplen_sec: 10                                                    
│       test_chunklen_sec: 10                                                   
│       test_hoplen_sec: 10                              

In [ ]:
import glob, shutil
files = glob.glob('logs/multi_accdoa_HTSAT/runs/*/checkpoints/last.ckpt')
shutil.copy(files[0], '/content/drive/MyDrive/PSELDNets_ckpts/last.ckpt')
print('saved:', files[0])

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
path = '/content/drive/MyDrive/PSELDNets_ckpts/last.ckpt'
mtime = os.path.getmtime(path)
import datetime
print(datetime.datetime.fromtimestamp(mtime))
print(f'size: {os.path.getsize(path)/1e6:.1f} MB')
